# DSPy Reasoning Module

This notebook demonstrates how to use the `dspy.Reasoning` module with reasoning-capable models.

**What You'll Learn:**
- How to use `dspy.Reasoning` with reasoning models
- How to capture and inspect reasoning traces
- How to use reasoning for complex problem solving
- How to combine reasoning with other DSPy modules
- How `Reasoning` differs from `ChainOfThought`

## Setup

Import libraries and configure the language model. For best results with reasoning, use models like GPT-4o or Claude 3.7 Sonnet (extended thinking).

In [ ]:
import os
import sys
sys.path.append('../../')

import dspy
from utils import setup_default_lm, print_step, print_result, print_error, configure_dspy
from dotenv import load_dotenv

load_dotenv('../../.env')

print_step("Setting up Language Model", "Configuring DSPy with a reasoning-capable model")

try:
    lm = setup_default_lm(provider="openai", model="gpt-4o", max_tokens=2000)
    configure_dspy(lm=lm)
    print_result("Language model configured successfully!", "Status")
except Exception as e:
    print_error(f"Failed to configure language model: {e}")
    print("Make sure you have set your OPENAI_API_KEY in the .env file")

## Example 1: Basic Reasoning

Use `dspy.Reasoning` to solve a complex logic problem with visible reasoning steps. The Reasoning module captures the model's internal thinking process.

In [ ]:
class ComplexProblemSolver(dspy.Signature):
    """Solve a complex problem with detailed reasoning."""
    problem = dspy.InputField(desc="A complex problem requiring multi-step reasoning")
    reasoning = dspy.OutputField(desc="Step-by-step reasoning and thought process")
    solution = dspy.OutputField(desc="The final solution to the problem")

reasoning_solver = dspy.Reasoning(ComplexProblemSolver)

problem1 = """
Three friends - Alice, Bob, and Carol - are standing in a line.
- Alice is not at the front.
- Bob is not at the back.
- Carol is not in the middle.
What is the order of the three friends from front to back?
"""

result1 = reasoning_solver(problem=problem1)
print_result(
    f"Problem: {problem1.strip()}\n\n"
    f"Reasoning Process:\n{result1.reasoning}\n\n"
    f"Solution: {result1.solution}",
    "Logic Problem"
)

## Example 2: Mathematical Reasoning

Solve complex math problems with detailed reasoning steps. Note how the model works through the problem systematically.

In [ ]:
class MathReasoning(dspy.Signature):
    """Solve a mathematical problem with detailed reasoning."""
    problem = dspy.InputField(desc="A mathematical problem to solve")
    reasoning = dspy.OutputField(desc="Mathematical reasoning and steps")
    answer = dspy.OutputField(desc="The numerical answer")

math_reasoner = dspy.Reasoning(MathReasoning)

math_problem = """
A train travels from City A to City B at 60 mph. On the return trip,
it travels at 40 mph. What is the average speed for the entire round trip?
(Note: This is a trick question - the answer is NOT 50 mph!)
"""

result2 = math_reasoner(problem=math_problem)
print_result(
    f"Problem: {math_problem.strip()}\n\n"
    f"Reasoning:\n{result2.reasoning}\n\n"
    f"Answer: {result2.answer}",
    "Math Problem"
)

## Example 3: Advanced Reasoning Module

Build a custom module that uses reasoning for problem decomposition: break down complex problems into sub-problems, solve each with reasoning, and combine the results.

In [ ]:
class ProblemDecomposer(dspy.Module):
    """Breaks down complex problems into steps, solves each with reasoning, and combines results."""
    def __init__(self):
        super().__init__()

        class DecomposeProblem(dspy.Signature):
            """Break down a complex problem into simpler sub-problems."""
            problem = dspy.InputField(desc="A complex problem")
            sub_problems = dspy.OutputField(desc="List of 2-4 simpler sub-problems, one per line")

        class SolveSubProblem(dspy.Signature):
            """Solve a sub-problem with reasoning."""
            sub_problem = dspy.InputField(desc="A sub-problem to solve")
            reasoning = dspy.OutputField(desc="Reasoning process")
            solution = dspy.OutputField(desc="Solution to the sub-problem")

        class CombineSolutions(dspy.Signature):
            """Combine sub-problem solutions into a final answer."""
            original_problem = dspy.InputField(desc="The original problem")
            solutions = dspy.InputField(desc="Solutions to all sub-problems")
            final_answer = dspy.OutputField(desc="The comprehensive final answer")

        self.decompose = dspy.Predict(DecomposeProblem)
        self.solve_step = dspy.Reasoning(SolveSubProblem)
        self.combine = dspy.Predict(CombineSolutions)

    def forward(self, problem):
        decomposition = self.decompose(problem=problem)
        sub_problems = [sp.strip() for sp in decomposition.sub_problems.split('\n') if sp.strip()]

        print(f"\n  Decomposed into {len(sub_problems)} sub-problems:")
        for i, sp in enumerate(sub_problems, 1):
            print(f"    {i}. {sp}")

        solutions = []
        for i, sub_problem in enumerate(sub_problems, 1):
            print(f"\n  Solving sub-problem {i}...")
            solution = self.solve_step(sub_problem=sub_problem)
            solutions.append(f"Sub-problem {i}: {sub_problem}\nReasoning: {solution.reasoning}\nSolution: {solution.solution}")

        combined_solutions = "\n\n".join(solutions)
        final = self.combine(original_problem=problem, solutions=combined_solutions)

        return dspy.Prediction(
            sub_problems=sub_problems,
            solutions=solutions,
            final_answer=final.final_answer
        )

decomposer = ProblemDecomposer()

complex_problem = """
A company wants to organize a team-building event for 150 employees.
They have a budget of $7,500. They need to:
1. Choose a venue (indoor or outdoor)
2. Arrange catering (breakfast, lunch, and snacks)
3. Plan 3 team activities
4. Arrange transportation if needed

Create a comprehensive plan that stays within budget and maximizes employee engagement.
"""

print("Starting problem decomposition and solving...")
result3 = decomposer(problem=complex_problem.strip())

print_result(
    f"Final Comprehensive Solution:\n{result3.final_answer}",
    "Complex Planning Problem"
)

## Example 4: Reasoning vs ChainOfThought

Compare `dspy.Reasoning` and `dspy.ChainOfThought` to understand the differences:
- **Reasoning**: Designed for reasoning-capable models (o1, Claude thinking). Captures native reasoning traces.
- **ChainOfThought**: Works with all models. Prompts the model to show reasoning via instruction.

In [ ]:
class ProblemSolver(dspy.Signature):
    """Solve a problem."""
    problem = dspy.InputField()
    reasoning = dspy.OutputField()
    answer = dspy.OutputField()

reasoning_module = dspy.Reasoning(ProblemSolver)
cot_module = dspy.ChainOfThought(ProblemSolver)

test_problem = "If you're in a race and you pass the person in 2nd place, what place are you in?"

print("Using dspy.Reasoning:")
reasoning_result = reasoning_module(problem=test_problem)
print(f"  Reasoning: {reasoning_result.reasoning[:200]}...")
print(f"  Answer: {reasoning_result.answer}")

print("\nUsing dspy.ChainOfThought:")
cot_result = cot_module(problem=test_problem)
print(f"  Reasoning: {cot_result.reasoning[:200]}...")
print(f"  Answer: {cot_result.answer}")

print("\nKey Differences:")
print("- dspy.Reasoning: Captures native reasoning from reasoning-capable models")
print("- dspy.ChainOfThought: Prompts model to show reasoning via instruction")

## Summary

**Key Takeaways:**
1. `dspy.Reasoning` captures native reasoning from reasoning-capable models
2. Use it for complex problems that benefit from detailed thinking
3. The reasoning trace is visible and can be used for debugging/validation
4. Combine with custom modules for advanced problem-solving workflows
5. For best results, use with models that support native reasoning (o1, Claude)